# 2º Workshop Eixo de Inovação — Retrieval-Augmented Generation (RAG)

**Rede Feminista em IA para a América Latina e o Caribe**

---

### Pré-requisitos
- Python >= 3.10
- `jupyterlab` (ou similar) instalado para executar este notebook

**Todo o resto é instalado automaticamente na primeira célula de código.**

### Modelos do workshop (toda a família Qwen: um único ecossistema coerente, multilíngue e aberto)
- **LLM default:** `Qwen/Qwen3-1.7B` — **Fallback (menos recursos):** `Qwen/Qwen3-0.6B`
- **Embeddings:** `Qwen/Qwen3-Embedding-0.6B`
- **Reranker:** `Qwen/Qwen3-Reranker-0.6B`

### Stack
- **Hugging Face** → modelos e datasets
- **LlamaIndex** → pipeline RAG e índice in-memory
- **Qdrant** → banco de dados vetorial (modo embutido, sem servidor nem Docker)

In [ ]:
import sys
print(f"Versão de Python em uso: {sys.version}")

## 0. Caminhos segundo nível e hardware

### 0.1 Objetivo final
Construir um **pipeline RAG sobre documentos próprios** de cada projeto, com respostas citadas e avaliação básica de qualidade.

### 0.2 Escolha seu caminho

| Caminho | Seções | Descrição |
|---------|--------|-----------|
| 🟢 Exploração simples | 1 → 2 → 3 → 6 → 8 → 12 | Setup → intuição RAG → embeddings → índice in-memory → pipeline RAG → aplicação |
| 🟡 Experimentação | 1 → 2 → 3 → 4 → 5 → 6 → 8 → 10 → 12 | + similaridade → chunking → citações |
| 🔴 Profundidade | 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 | Fluxo completo: + Qdrant → reranking → avaliação |
| ⚙️ Hardware limitado | 1 → 2 → 3 → 6 → 8 → 10 → 12 | Mesmo fluxo com Qwen3-0.6B; o retrieval é leve, o gargalo é o LLM |

### 0.3 Escolha segundo seu hardware
- 🖥️ **Somente CPU** → Embeddings: `Qwen3-Embedding-0.6B` roda em CPU (indexar demora mais, mas o corpus é pequeno). LLM: `Qwen3-0.6B`. Qdrant embutido ou índice in-memory.
- ⚡ **GPU disponível** → Fluxo completo com `Qwen3-1.7B` + reranker (seção 9).
- ❓ **Não sabe que hardware tem** → Avance até **1.4 (detecção de hardware)** e decida lá.

### 0.4 Como usar este notebook
- ▶️ **Executar esta célula** → base funcional imediata
- 🧪 **Experimente isto** → modificar e experimentar
- 🚀 **Vá além** → aprofundamento opcional

---
## 1. Setup local (adaptado ao hardware)

### 1.1 Ambiente
Recomendamos executar este notebook dentro de um ambiente virtual (`venv` ou `conda`) para não misturar dependências com outros projetos:

```bash
python -m venv .venv
source .venv/bin/activate      # Linux / macOS
pip install jupyterlab
jupyter lab
```

### 1.2 Instalação base
A célula seguinte (▶️) instala tudo o que precisamos:

| Pacote | Papel no pipeline |
|--------|-------------------|
| `transformers`, `sentence-transformers` | carregar os modelos do Hugging Face (LLM, embeddings, reranker) |
| `llama-index-core` + integrações | orquestrar o pipeline RAG (documentos, índice, query engine) |
| `qdrant-client` | banco de dados vetorial embutido (seção 7) — instala com pip e pronto, sem servidor nem Docker |
| `plotly`, `scikit-learn` | visualizações interativas e projeções 2D |

In [ ]:
# ═══ 1.2 Instalação base ═══
import subprocess
import sys

# Garantir que pip esteja disponível
try:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "--version"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
except subprocess.CalledProcessError:
    print("⏳ Instalando pip...")
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--default-pip"])

pacotes = [
    "torch",
    "transformers",
    "sentence-transformers",
    "accelerate",
    "huggingface_hub",
    "llama-index-core",
    "llama-index-embeddings-huggingface",
    "llama-index-llms-huggingface",
    "llama-index-vector-stores-qdrant",
    "qdrant-client",
    "numpy",
    "scikit-learn",
    "pandas",
    "plotly",
]

print("📦 Instalando pacotes necessários (pode demorar alguns minutos na primeira vez)...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pacotes)
print("✅ Pacotes instalados corretamente")

In [ ]:
# ═══ Imports e configuração geral ═══
import gc
import json
import os
import re
import time
import warnings

import numpy as np
import torch

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ═══ Modelos do workshop ═══
NOME_LLM_DEFAULT = "Qwen/Qwen3-1.7B"        # requer GPU ou >=16GB RAM
NOME_LLM_FALLBACK = "Qwen/Qwen3-0.6B"       # funciona bem em CPU / hardware limitado
NOME_EMBEDDINGS = "Qwen/Qwen3-Embedding-0.6B"
NOME_RERANKER = "Qwen/Qwen3-Reranker-0.6B"

# ═══ Paleta para as visualizações do workshop ═══
CORES = ["#2a78d6", "#eb6834", "#1baf7a"]
ESCALA_SEQUENCIAL = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
COR_TINTA = "#0b0b0b"
COR_GRADE = "#e1e0d9"

import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

def estilizar_figura(fig, titulo=None):
    """Aplica o estilo comum do workshop a uma figura do plotly."""
    fig.update_layout(
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", color=COR_TINTA, size=13),
        paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb",
        title=titulo, margin=dict(l=60, r=30, t=60 if titulo else 30, b=60),
    )
    fig.update_xaxes(gridcolor=COR_GRADE, zerolinecolor=COR_GRADE)
    fig.update_yaxes(gridcolor=COR_GRADE, zerolinecolor=COR_GRADE)
    return fig

print("✅ Configuração pronta")

### 1.3 Qdrant embutido

Qdrant é o banco de dados vetorial que vamos usar na seção 7. A boa notícia: ele tem um **modo local embutido** que guarda tudo em um arquivo no disco. Não é preciso subir um servidor nem instalar Docker — `pip install qdrant-client` (que já fizemos acima) é todo o setup.

### 1.4 Detecção de hardware
A célula seguinte detecta seu hardware e escolhe o LLM apropriado. O **retrieval** (embeddings + busca) é leve e roda bem em qualquer máquina; o que define o quão confortável vai ser o workshop é o **LLM gerador**.

In [ ]:
# ═══ 1.4 Detecção de hardware ═══
import platform

print("=" * 55)
print("🔍 DETECÇÃO DE HARDWARE")
print("=" * 55)

print(f"\n💻 Sistema: {platform.system()} {platform.machine()}")
print(f"   Python: {sys.version.split()[0]}")

# RAM
if platform.system() == "Darwin":
    ram_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
elif platform.system() == "Linux":
    ram_bytes = 0
    with open("/proc/meminfo") as f:
        for linha in f:
            if "MemTotal" in linha:
                ram_bytes = int(linha.split()[1]) * 1024
                break
else:
    ram_bytes = 0

ram_gb = ram_bytes / (1024**3) if ram_bytes else 0
if ram_gb > 0:
    print(f"   RAM total: {ram_gb:.1f} GB")

# GPU / MPS
tem_cuda = torch.cuda.is_available()
tem_mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

if tem_cuda:
    nome_gpu = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    dispositivo = "cuda"
    print(f"\n⚡ GPU detectada: {nome_gpu} ({vram_gb:.1f} GB VRAM)")
elif tem_mps:
    dispositivo = "mps"
    vram_gb = ram_gb  # memória unificada
    print("\n⚡ GPU detectada: 🍎 Apple Silicon (backend MPS)")
else:
    dispositivo = "cpu"
    vram_gb = 0
    print("\n🖥️ Somente CPU disponível")

# Escolha automática do LLM (você pode sobrescrevê-la à mão)
if (tem_cuda and vram_gb >= 6) or ram_gb >= 16:
    nome_llm = NOME_LLM_DEFAULT
else:
    nome_llm = NOME_LLM_FALLBACK

print("\n" + "=" * 55)
print(f"✅ Dispositivo escolhido: {dispositivo}")
print(f"✅ LLM escolhido:         {nome_llm}")
print(f"   Embeddings:            {NOME_EMBEDDINGS}")
print(f"   Reranker (seção 9):    {NOME_RERANKER}")
print("=" * 55)

# 🧪 Experimente isto: forçar o outro modelo descomentando a linha que quiser
# nome_llm = NOME_LLM_DEFAULT
# nome_llm = NOME_LLM_FALLBACK

### 1.5 Corpus do workshop

Para que todas as pessoas trabalhem sobre os mesmos documentos, a célula seguinte cria a pasta `corpus_workshop/` com os documentos de uma **biblioteca comunitária fictícia** (a *Biblioteca Comunitária Arroyo Claro*), em espanhol e português: regulamentos, atas, guias e FAQs.

Por que um corpus inventado? Porque assim podemos verificar com certeza qual informação **está** e qual informação **não está** nos documentos — algo fundamental quando avaliarmos alucinações (seção 10). Além disso, como é fictício, o modelo não pode "lembrar" de nada: **tudo o que ele responder certo tem que sair do retrieval**.

> 🧪 **Experimente isto (para seu projeto):** crie uma pasta `meus_documentos/` com 5–10 documentos próprios (`.txt` ou `.md`) e use-a na seção 12.

In [ ]:
# ═══ 1.5 Criar o corpus do workshop ═══
from pathlib import Path

PASTA_CORPUS = Path("corpus_workshop")
PASTA_CORPUS.mkdir(exist_ok=True)

documentos_workshop = {
    "historia_biblioteca.md": """\
# Historia de la Biblioteca Comunitaria Arroyo Claro

La Biblioteca Comunitaria Arroyo Claro fue fundada en 2011 por un grupo de
vecinas y vecinos del barrio Arroyo Claro. Comenzó funcionando en un galpón
prestado, con una colección inicial de 300 libros donados.

En 2015 la biblioteca se mudó a su local actual, ubicado junto a la estación
de tren. Gracias a campañas de donación y a un convenio con editoriales
independientes, la colección creció hasta superar los 12.000 ejemplares.

La biblioteca es gestionada por una comisión de 7 personas, elegidas en
asamblea cada 2 años. Todas las actividades son organizadas por personas
voluntarias y el acceso a la biblioteca es libre y gratuito.
""",
    "reglamento_prestamos.md": """\
# Reglamento de préstamos

1. Pueden retirar materiales todas las personas asociadas a la biblioteca.
   Asociarse es gratuito: solo hace falta completar un formulario con datos
   de contacto.
2. Cada persona puede llevarse hasta 3 libros a la vez, por un plazo de
   21 días.
3. El préstamo puede renovarse una única vez, por 14 días más, siempre que
   el libro no haya sido reservado por otra persona. La renovación puede
   hacerse en el mostrador o por correo electrónico.
4. Los materiales de referencia (enciclopedias, diccionarios y mapas) no se
   prestan a domicilio: se consultan únicamente en la sala de lectura.
5. En caso de atraso en la devolución, se suspende la posibilidad de retirar
   nuevos materiales por el doble de días del atraso.
6. La cuota social es voluntaria y se destina al mantenimiento del edificio
   y a la compra de nuevos materiales.
""",
    "talleres_y_actividades.md": """\
# Talleres y actividades regulares

- **Taller de huerta**: sábados a las 10:00, en el patio trasero. Abierto a
  todas las edades. Coordinado por el grupo de la huerta comunitaria.
- **Club de lectura**: primer jueves de cada mes a las 18:00, en la sala de
  lectura. Cada mes se vota el libro siguiente.
- **Taller de ajedrez**: miércoles a las 17:00. A partir de los 8 años.
  Hay tableros disponibles en la biblioteca, no hace falta traer.
- **Apoyo escolar**: lunes y viernes de 16:00 a 18:00, para estudiantes de
  primaria y secundaria.

La inscripción a todos los talleres es gratuita y puede hacerse en el
mostrador de la biblioteca o escribiendo por correo electrónico.
""",
    "guia_huerta_semillas.md": """\
# Guía del banco de semillas

La biblioteca mantiene un banco de semillas comunitario desde 2018, junto
al grupo de la huerta.

## ¿Cómo funciona el intercambio?

Cualquier persona puede llevarse hasta 3 sobres de semillas por temporada.
El compromiso es simple: después de la cosecha, devolver al banco el doble
de las semillas retiradas, para que el banco siga creciendo.

## Variedades disponibles

- Tomate platense
- Zapallo anco
- Albahaca de hoja ancha
- Acelga

## Calendario breve de siembra

- **Primavera**: tomate, albahaca y zapallo.
- **Otoño**: acelga y otras verduras de hoja.

Las semillas se entregan en sobres de papel etiquetados con la variedad y
la fecha de cosecha.
""",
    "acervo_emprestimos_pt.md": """\
# Acervo em português

A biblioteca possui uma seção com cerca de 800 títulos em português,
principalmente de literatura brasileira contemporânea, formada a partir de
um intercâmbio com uma biblioteca comunitária de Porto Alegre iniciado
em 2019.

As regras de empréstimo são as mesmas do regulamento geral: até 3 livros
por vez, por 21 dias.

Doações de livros em português são recebidas às sextas-feiras, das 14:00
às 18:00, diretamente no balcão. Antes de doar coleções grandes (mais de
50 livros), pedimos que entre em contato por correio eletrônico para
coordenar a entrega.
""",
    "oficina_compostagem_pt.md": """\
# Oficina de compostagem

A oficina de compostagem acontece na última terça-feira de cada mês, às
17:00, no pátio da biblioteca. A participação é gratuita e aberta a todas
as pessoas.

A biblioteca mantém três composteiras no pátio, que recebem os restos
orgânicos da copa e das atividades.

## O que pode ir na composteira

Cascas de frutas e legumes, borra de café, folhas secas, papel picado
sem tinta colorida.

## O que não pode ir na composteira

Carnes, laticínios, gorduras, frutas cítricas em grande quantidade e
qualquer material plástico.

O composto produzido é usado na horta comunitária e distribuído entre as
pessoas participantes da oficina.
""",
    "equipamiento_sala_digital.md": """\
# Sala digital: equipamiento y uso

La sala digital de la biblioteca cuenta con:

- 6 computadoras de escritorio con acceso a internet.
- 1 impresora 3D, donada en 2022 por una cooperativa tecnológica del barrio.
- Conexión wifi abierta, red "BiblioArroyo", disponible en todo el edificio.

## Uso de la impresora 3D

La impresora 3D puede reservarse con al menos 48 horas de anticipación en
el mostrador. Cada persona puede reservar hasta 4 horas de impresión por
semana. El material de impresión (filamento PLA) lo aporta la biblioteca
para proyectos comunitarios y educativos.

## Cursos

Los martes a las 18:00 se dicta el curso de alfabetización digital, de
nivel inicial, sin requisitos previos.
""",
    "acta_asamblea_2024.md": """\
# Acta de la asamblea general — marzo de 2024

Resumen de las decisiones tomadas por la asamblea general de personas
asociadas, realizada en marzo de 2024:

1. **Nuevo horario de atención**: la biblioteca abrirá de lunes a sábado,
   de 9:00 a 19:00. Antes cerraba a las 17:00.
2. **Fondo de reparación de techos**: se crea un fondo específico para
   reparar los techos de la sala de lectura, con lo recaudado en la feria
   anual del libro usado.
3. **Ampliación de la comisión**: se incorporan 2 personas a la comisión
   directiva, que pasa de 7 a 9 integrantes hasta la próxima elección.
4. **Próxima asamblea**: se convocará en marzo de 2025.
""",
}

for nome_arquivo, conteudo in documentos_workshop.items():
    (PASTA_CORPUS / nome_arquivo).write_text(conteudo, encoding="utf-8")

print(f"✅ Corpus criado em '{PASTA_CORPUS}/' com {len(documentos_workshop)} documentos:")
for nome_arquivo in documentos_workshop:
    tamanho = len((PASTA_CORPUS / nome_arquivo).read_text(encoding="utf-8"))
    idioma = "PT" if nome_arquivo.endswith("_pt.md") else "ES"
    print(f"   [{idioma}] {nome_arquivo:<32} ({tamanho} caracteres)")

---
## 2. RAG: contexto mínimo

### 2.1 O problema: o que o modelo não sabe

Um LLM tem o conhecimento **congelado no momento do seu treinamento**. Por construção, ele não sabe nada sobre:

- **Dados privados ou locais**: os documentos internos da sua organização, atas, entrevistas, bases de conhecimento comunitárias.
- **Dados recentes**: qualquer coisa posterior à data de corte do treinamento.
- **Dados específicos demais**: mesmo que tenham estado na internet, detalhes de nicho podem não ter ficado "gravados".

E o mais problemático: quando perguntamos algo que ele não sabe, muitas vezes **não diz "não sei"** — inventa uma resposta plausível. Isso é uma **alucinação** (vimos no workshop anterior, seção 8.5).

Nosso corpus da biblioteca Arroyo Claro é fictício, então é um caso extremo e perfeito: o modelo não pode saber *nada* sobre ele. Qualquer resposta correta vai ter que sair dos documentos.

### 2.2 A ideia central de RAG

**Buscar primeiro, gerar depois.**

Em vez de pedir para o LLM responder "de memória", o pipeline RAG:

1. **Busca** nos nossos documentos os fragmentos mais relevantes para a pergunta.
2. **Monta um prompt** que inclui esses fragmentos como contexto.
3. **Gera** a resposta usando esse contexto recuperado, não só a memória do modelo.

O LLM deixa de ser um "oráculo que sabe tudo" e passa a ser um **redator que lê e sintetiza** o que aproximamos dele.

### 2.3 Anatomia de um pipeline RAG

Este é o diagrama de referência que vamos construir **peça por peça** no resto do notebook:

```
  INDEXAÇÃO (uma única vez, offline)
  ┌──────────────┐   ┌──────────┐   ┌────────────┐   ┌──────────────────┐
  │  documentos  │ → │ chunking │ → │ embeddings │ → │  índice vetorial │
  │  (seção 1)   │   │ (seção 5)│   │  (seção 3) │   │  (seções 6 e 7)  │
  └──────────────┘   └──────────┘   └────────────┘   └──────────────────┘

  CONSULTA (cada vez que alguém pergunta)
  ┌──────────┐   ┌───────────┐   ┌─────────────┐   ┌────────────────┐
  │ pergunta │ → │ retrieval │ → │ (reranking) │ → │  geração       │
  │          │   │ (seç. 4-6)│   │  (seção 9)  │   │  com citações  │
  └──────────┘   └───────────┘   └─────────────┘   │  (seç. 8 e 10) │
                                                   └────────────────┘
```

### 2.4 RAG vs fine-tuning vs prompt longo

| | **RAG** | **Fine-tuning** | **Prompt longo** (tudo no prompt) |
|---|---|---|---|
| O que faz? | busca e anexa contexto relevante | ajusta os pesos do modelo | cola todos os documentos no prompt |
| Atualizar conhecimento | adicionar/editar documentos, sem retreinar | retreinar a cada vez | editar o prompt |
| Custo | baixo (indexar é barato) | alto (GPU, dados, tempo) | cresce a cada consulta (tokens) |
| Rastreabilidade | alta: dá para citar a fonte | nula: o conhecimento fica "diluído" nos pesos | média |
| Limite principal | qualidade do retrieval | precisa de muitos exemplos | janela de contexto do modelo |
| Ideal para | conhecimento factual que muda | **estilo, formato, tom** | corpus muito pequenos (< algumas páginas) |

Regra prática: **RAG para conhecimento, fine-tuning para comportamento.** E dá para combinar os dois.

### 2.5 Casos de uso aplicados aos projetos

- **Bases de conhecimento comunitárias**: responder perguntas frequentes sobre o funcionamento de uma organização, com fontes citadas.
- **Documentação interna**: regulamentos, manuais, atas de reunião que ninguém tem tempo de reler.
- **Arquivos e entrevistas**: tornar navegável um acervo de história oral ou de entrevistas transcritas.
- **Normativas**: encontrar o que uma norma ou um regulamento diz sobre um tema específico, com referência ao texto original.

> 💬 **Para discutir com sua equipe:** quais documentos do seu projeto as pessoas consultam de novo e de novo? Essa é sua primeira candidata a corpus RAG.

---
## 3. Embeddings: representar significado com vetores

### 3.1 O que é um embedding

Um **embedding** converte um texto em um **vetor de números** (uma lista de, por exemplo, 1024 valores). A propriedade mágica: textos com **significado parecido** ficam **próximos no espaço** vetorial, mesmo sem compartilhar nenhuma palavra.

```
  "A que horas a biblioteca abre?"    →  [ 0.12, -0.48, 0.33, ... ]   ┐
  "horário de atendimento ao público" →  [ 0.10, -0.45, 0.30, ... }   ├─ perto
                                                                      ┘
  "receita de pão caseiro"            →  [-0.52,  0.07, 0.81, ... ]   ← longe
```

Essa intuição geométrica — **perto no espaço = parecido em significado** — é o coração de todo o retrieval que vem a seguir.

### 3.2 Carregar um modelo de embeddings (Hugging Face)

Vamos usar `Qwen/Qwen3-Embedding-0.6B` através da biblioteca `sentence-transformers`, que nos dá uma interface simples (`.encode()`). A alternativa é usar `transformers` "cru" e fazer o pooling à mão — mais controle, mais código; para este workshop não é necessário.

In [ ]:
# ═══ 3.2 Carregar o modelo de embeddings ═══
from sentence_transformers import SentenceTransformer

print(f"⏳ Carregando modelo de embeddings: {NOME_EMBEDDINGS}")
print("   (na primeira vez baixa os pesos, ~1.2 GB)")

modelo_embeddings = SentenceTransformer(NOME_EMBEDDINGS, device=dispositivo)

dimensao = modelo_embeddings.get_embedding_dimension()
print(f"✅ Modelo carregado")
print(f"   Dimensão do vetor: {dimensao}")
print(f"   Parâmetros: ~0.6B — pequeno para um LLM, potente para um embedder")

### 3.3 Primeiro exercício: gerar embeddings de frases

▶️ Vamos converter algumas frases em vetores e olhar o que tem dentro.

In [ ]:
# ═══ 3.3 Gerar embeddings de frases e inspecionar o vetor ═══
frases = [
    "Quantos livros posso levar da biblioteca?",
    "Regulamento de empréstimo de materiais",
    "A oficina de horta é aos sábados de manhã",
    "Receita de pão caseiro com fermentação natural",
]

vetores = modelo_embeddings.encode(frases)

print(f"Quantidade de frases: {len(frases)}")
print(f"Shape da matriz de vetores: {vetores.shape}")
print(f"  → {vetores.shape[0]} frases, cada uma representada por {vetores.shape[1]} números\n")

print(f"Primeiros 8 valores do vetor da frase 1:")
print(f"  {np.round(vetores[0][:8], 4)}")
print(f"\nNorma (comprimento) do vetor 1: {np.linalg.norm(vetores[0]):.4f}")
print("  → sentence-transformers já devolve vetores normalizados (norma ≈ 1)")

# 🧪 Experimente isto: troque as frases por frases do SEU projeto e execute de novo

### 3.4 Embeddings multilíngues e equidade

No workshop anterior vimos que a **tokenização é desigual entre idiomas** (a mesma frase "custa" mais tokens em guarani do que em inglês). Com os embeddings acontece algo análogo: os modelos são treinados majoritariamente com inglês (e chinês, no caso do Qwen), e a qualidade da representação **se degrada em línguas menos representadas**.

O teste decisivo: frases **equivalentes** em idiomas diferentes caem **perto** no espaço vetorial?

In [ ]:
# ═══ 3.4 Frases equivalentes em idiomas diferentes caem perto? ═══
frases_equivalentes = {
    "português": "A água está fria",
    "español":   "El agua está fría",
    "english":   "The water is cold",
    "guarani":   "Y iro'ysã",
}
frase_controle = "A impressora 3D é reservada com 48 horas de antecedência"

textos = list(frases_equivalentes.values()) + [frase_controle]
rotulos = list(frases_equivalentes.keys()) + ["controle (outro tema)"]
vecs = modelo_embeddings.encode(textos)

print("Similaridade cosseno contra a frase em PORTUGUÊS:\n")
for i in range(1, len(textos)):
    sim = float(vecs[0] @ vecs[i])  # vetores normalizados: produto escalar = cosseno
    barra = "█" * int(sim * 40)
    print(f"  {rotulos[i]:<22} {sim:.3f}  {barra}")

print("""
📝 O que observar:
  - espanhol e inglês costumam ficar MUITO perto do português (idiomas bem
    representados no treinamento)
  - guarani costuma ficar mais longe, mesmo sendo a MESMA ideia: o modelo
    o viu pouco durante o treinamento
  - a frase de controle (outro tema, mesmo idioma) deveria ficar mais longe
    que as traduções... isso também vale para o guarani?
""")

> 💬 **Para discutir:** se o seu corpus está em uma língua ou variedade pouco representada, o retrieval vai ser menos preciso — e isso é **invisível** se você só testa em português. Como você detectaria isso no seu projeto? (Dica: seção 11, avaliação.)

### 3.5 Visualização: o mapa do espaço semântico

Os vetores têm 1024 dimensões — impossíveis de desenhar. Mas podemos **projetá-los em 2D** com PCA (análise de componentes principais) para ver sua organização. A projeção perde informação, mas conserva a estrutura geral: os **clusters temáticos**.

▶️ A visualização é **interativa**: passe o mouse por cada ponto para ver a frase completa.

In [ ]:
# ═══ 3.5 Projetar embeddings em 2D e inspecionar clusters ═══
from sklearn.decomposition import PCA

grupos = {
    "biblioteca": [
        "Quantos livros posso pegar emprestado?",
        "A renovação é pedida no balcão",
        "Os dicionários são consultados na sala de leitura",
        "O horário de atendimento é das 9 às 19",
        "La biblioteca presta hasta tres libros",
    ],
    "horta": [
        "Na primavera se plantam tomates e manjericão",
        "O composto é preparado com restos de legumes",
        "As sementes são guardadas em envelopes de papel",
        "A acelga é colhida no outono",
        "El zapallo se cosecha en otoño",
    ],
    "tecnologia": [
        "A impressora 3D usa filamento PLA",
        "A sala tem seis computadores com internet",
        "O wi-fi da biblioteca é uma rede aberta",
        "O curso de alfabetização digital é às terças",
        "La impresora 3D se reserva con anticipación",
    ],
}

todas_frases = [f for fs in grupos.values() for f in fs]
todos_grupos = [g for g, fs in grupos.items() for _ in fs]

vecs_corpus = modelo_embeddings.encode(todas_frases)
projecao = PCA(n_components=2).fit_transform(vecs_corpus)

fig = go.Figure()
for i, (nome_grupo, cor) in enumerate(zip(grupos, CORES)):
    mascara = [g == nome_grupo for g in todos_grupos]
    fig.add_trace(go.Scatter(
        x=projecao[mascara, 0], y=projecao[mascara, 1],
        mode="markers", name=nome_grupo,
        marker=dict(size=11, color=cor, line=dict(width=1, color="#fcfcfb")),
        text=[f for f, m in zip(todas_frases, mascara) if m],
        hovertemplate="%{text}<extra>" + nome_grupo + "</extra>",
    ))

estilizar_figura(fig, "Espaço semântico projetado em 2D (PCA) — passe o mouse pelos pontos")
fig.update_layout(xaxis_title="componente 1", yaxis_title="componente 2", height=480)
fig.show()

print("📝 Cada ponto é uma frase; a cor é o tema. Repare que:")
print("   - as frases do mesmo tema formam clusters, SEM compartilhar palavras")
print("   - as frases em espanhol caem dentro do cluster do seu tema:")
print("     o espaço é (majoritariamente) multilíngue")

# 🧪 Experimente isto: adicione um grupo com 5 frases do seu projeto e veja onde cai

---
## 4. Similaridade: medir proximidade entre vetores

### 4.1 Similaridade cosseno

A métrica padrão para comparar embeddings de texto é a **similaridade cosseno**: o cosseno do **ângulo** entre dois vetores.

$$\text{sim}(\vec{a}, \vec{b}) = \frac{\vec{a} \cdot \vec{b}}{\|\vec{a}\| \, \|\vec{b}\|}$$

- Mede **direção**, não magnitude: não importa o quão "longos" os vetores são, e sim para onde apontam.
- Intervalo: de -1 (opostos) a 1 (idênticos). Em embeddings de texto quase sempre cai entre 0 e 1.

▶️ A implementação são 3 linhas de numpy:

In [ ]:
# ═══ 4.1 Similaridade cosseno em 3 linhas ═══
def similaridade_cosseno(a, b):
    """Cosseno do ângulo entre dois vetores."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

v1, v2 = modelo_embeddings.encode(["olá, como você está?", "oi, tudo bem?"])
v3 = modelo_embeddings.encode(["o tomate é plantado na primavera"])[0]

print(f"'ola como voce esta' vs 'oi tudo bem':          {similaridade_cosseno(v1, v2):.3f}")
print(f"'ola como voce esta' vs 'o tomate e plantado':  {similaridade_cosseno(v1, v3):.3f}")

### 4.2 Outras métricas

| Métrica | Fórmula | Quando se usa |
|---------|---------|---------------|
| **Cosseno** | ângulo entre vetores | default em texto: o "comprimento" do vetor não carrega significado |
| **Produto escalar** | $\vec{a} \cdot \vec{b}$ | quando a magnitude importa (p. ex. popularidade em recomendação) |
| **Distância euclidiana** | $\|\vec{a} - \vec{b}\|$ | espaços onde a distância absoluta tem sentido físico |

Dado importante: se os vetores estão **normalizados** (norma = 1, como os nossos), o cosseno e o produto escalar **coincidem**, e a distância euclidiana ordena igual. Por isso os bancos vetoriais podem usar a métrica mais barata de calcular.

In [ ]:
# ═══ 4.2 Com vetores normalizados, cosseno == produto escalar ═══
a, b = modelo_embeddings.encode(["empréstimo de livros", "retirar materiais da biblioteca"])

print(f"Norma de a: {np.linalg.norm(a):.4f}   Norma de b: {np.linalg.norm(b):.4f}")
print(f"Similaridade cosseno: {similaridade_cosseno(a, b):.6f}")
print(f"Produto escalar:      {np.dot(a, b):.6f}")
print(f"Distância euclidiana: {np.linalg.norm(a - b):.4f}")
print(f"  → relação exata: dist² = 2 - 2·cosseno → {np.sqrt(2 - 2 * np.dot(a, b)):.4f}")

### 4.3 Exercício: matriz de similaridade com pares armadilha

Vamos montar uma matriz de similaridade entre frases escolhidas com malícia:

- **mesma palavra, sentido diferente**: "banco" (para sentar) vs "banco" (de sementes)
- **idioma diferente, mesmo sentido**: empréstimo de livros em PT e ES

▶️ A matriz é interativa: passe o mouse por cada célula.

In [ ]:
# ═══ 4.3 Matriz de similaridade ═══
frases_armadilha = [
    "Sentei no banco da praça",                      # banco: assento
    "O banco de sementes abre aos sábados",          # banco: instituição/coleção
    "Peguei três livros emprestados",                # empréstimo PT
    "Retiré tres libros en préstamo",                # préstamo ES (mesmo sentido)
    "A biblioteca empresta até três livros",         # empréstimo PT (paráfrase)
    "A abóbora é colhida no outono",                 # horta
]
rotulos_curtos = ["banco (praça)", "banco (sementes)", "empréstimo PT",
                  "préstamo ES", "empresta livros", "abóbora"]

vecs_armadilha = modelo_embeddings.encode(frases_armadilha)
matriz = vecs_armadilha @ vecs_armadilha.T  # normalizados: produto escalar = cosseno

fig = go.Figure(go.Heatmap(
    z=np.round(matriz, 2),
    x=rotulos_curtos, y=rotulos_curtos,
    colorscale=[[i / (len(ESCALA_SEQUENCIAL) - 1), cor] for i, cor in enumerate(ESCALA_SEQUENCIAL)],
    zmin=0, zmax=1,
    text=np.round(matriz, 2), texttemplate="%{text}",
    hovertemplate="%{y}<br>vs %{x}<br>similaridade: %{z}<extra></extra>",
    colorbar=dict(title="cosseno"),
))
estilizar_figura(fig, "Matriz de similaridade cosseno — pares armadilha")
fig.update_layout(height=520, yaxis_autorange="reversed")
fig.show()

print("📝 O que observar:")
print("   - 'empréstimo PT' vs 'préstamo ES' (idioma diferente, mesmo sentido): ALTA")
print("   - 'banco (praça)' vs 'banco (sementes)' (mesma palavra, outro sentido):")
print("     mais baixa que os pares de significado equivalente")
print("   - 'abóbora' fica longe de tudo o que é bibliotecário")

### 4.4 Busca semântica mínima (sem frameworks)

Com o que já temos, dá para construir **retrieval completo em ~10 linhas de numpy**: gerar embeddings do corpus, gerar o embedding da pergunta, ordenar por similaridade e ficar com o top-k.

Vale a pena entender esta célula **antes** de delegar o trabalho para LlamaIndex e Qdrant: por baixo, eles fazem exatamente isto (mais rápido e com mais organização).

In [ ]:
# ═══ 4.4 RAG-retrieval em 10 linhas ═══
# 1. Corpus: cada parágrafo de cada documento é uma unidade de busca
paragrafos, origens = [], []
for nome_arquivo in sorted(os.listdir(PASTA_CORPUS)):
    texto = (PASTA_CORPUS / nome_arquivo).read_text(encoding="utf-8")
    for paragrafo in texto.split("\n\n"):
        if paragrafo.strip():
            paragrafos.append(paragrafo.strip())
            origens.append(nome_arquivo)

# 2. Gerar embeddings de TODO o corpus (isto é feito UMA vez e salvo: é o "índice")
vecs_paragrafos = modelo_embeddings.encode(paragrafos)
print(f"Índice caseiro pronto: {len(paragrafos)} parágrafos com embeddings\n")

# 3. Buscar: gerar o embedding da pergunta e ordenar por similaridade
def buscar(pergunta, k=3):
    vec_pergunta = modelo_embeddings.encode([pergunta])[0]
    similaridades = vecs_paragrafos @ vec_pergunta
    top_k = np.argsort(similaridades)[::-1][:k]
    return [(similaridades[i], origens[i], paragrafos[i]) for i in top_k]

for sim, origem, paragrafo in buscar("quantos livros posso levar?"):
    print(f"[{sim:.3f}] ({origem})")
    print(f"   {paragrafo}...\n")

# 🧪 Experimente isto: busque em espanhol ("¿cuándo puedo donar libros?") e veja
# se encontra o documento correto mesmo com o índice majoritariamente em espanhol

### 4.5 Discussão: similaridade ≠ relevância

A busca por vizinhos mais próximos tem limites que convém conhecer desde o primeiro dia:

- **Similaridade temática não é relevância**: para "quantos livros posso levar?", um parágrafo que diz "os livros são maravilhosos" pode ser muito *similar* e nada *relevante*.
- **Perguntas com negação ou condições** ("o que NÃO pode ir na composteira?") se parecem muito com o seu oposto.
- **Informação espalhada**: se a resposta requer combinar dois parágrafos distantes, o top-k pode trazer só um.

Parte dessas limitações é atacada com **reranking** (seção 9) e com bom **chunking** (seção 5, agora mesmo).

---
## 5. Chunking: dividir documentos em pedaços úteis

### 5.1 Por que dividir os documentos

Duas razões:

1. **Limite de contexto do LLM**: não dá para colocar 12.000 livros (nem 12 documentos longos) em um prompt.
2. **Precisão do retrieval**: o embedding de um documento inteiro "faz a média" de todos os seus temas.
   - Chunks **grandes** → diluem: o trecho relevante fica enterrado entre parágrafos que não são.
   - Chunks **pequenos** → descontextualizam: "são emprestados por 21 dias" sem saber *o que* é emprestado.

### 5.2 Estratégias

- **Tamanho fixo** (por tokens ou caracteres): simples, ignora a estrutura.
- **Por sentenças** (`SentenceSplitter` do LlamaIndex): corta tentando respeitar limites de sentença. É o default razoável.
- **Por estrutura** (títulos, parágrafos, markdown): ideal quando os documentos têm estrutura clara (`MarkdownNodeParser`).

### 5.3 Parâmetros-chave

- `chunk_size`: tamanho alvo do chunk (em tokens).
- `chunk_overlap`: quantos tokens se repetem entre chunks consecutivos, para não cortar ideias ao meio.
- **Metadata que viaja com cada chunk**: fonte, título, página. É o que depois permite **citar** (seção 10) e **filtrar** (seção 7.5).

In [ ]:
# ═══ 5.2/5.3 SentenceSplitter em ação ═══
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Document

texto_exemplo = (PASTA_CORPUS / "reglamento_prestamos.md").read_text(encoding="utf-8")
doc_exemplo = Document(text=texto_exemplo, metadata={"file_name": "reglamento_prestamos.md"})

separador = SentenceSplitter(chunk_size=64, chunk_overlap=10)
chunks = separador.get_nodes_from_documents([doc_exemplo])

print(f"Documento original: {len(texto_exemplo)} caracteres")
print(f"Com chunk_size=64 tokens e overlap=10 → {len(chunks)} chunks\n")
print("═" * 60)
for i, chunk in enumerate(chunks[:3]):
    print(f"CHUNK {i} (metadata: {chunk.metadata.get('file_name')})")
    print(chunk.get_content())
    print("═" * 60)
print("(são mostrados apenas os 3 primeiros)")

### 5.4 Exercício: 3 configurações, mesmo documento, mesma pergunta

Vamos fazer o chunking de **todo o corpus** com três configurações diferentes e comparar **o que cada uma recupera** para a mesma pergunta, usando nosso buscador caseiro da seção 4.4.

In [ ]:
# ═══ 5.4 Comparar 3 configurações de chunking ═══
from llama_index.core import SimpleDirectoryReader

# Nota: a metadata viaja DENTRO do chunk ao gerar o embedding, então deixamos
# apenas o nome do arquivo (a default do reader ocupa ~32 tokens por chunk)
documentos = SimpleDirectoryReader(
    str(PASTA_CORPUS),
    file_metadata=lambda caminho: {"file_name": os.path.basename(caminho)},
).load_data()

configuracoes = {
    "pequeno (64 tokens)": SentenceSplitter(chunk_size=64, chunk_overlap=0),
    "médio (128 tokens)": SentenceSplitter(chunk_size=128, chunk_overlap=20),
    "grande (512 tokens)": SentenceSplitter(chunk_size=512, chunk_overlap=50),
}

PERGUNTA_TESTE = "Por quantos dias um empréstimo pode ser renovado?"

resultados_config = {}
for nome_config, splitter in configuracoes.items():
    nos_chunk = splitter.get_nodes_from_documents(documentos)
    textos_nos = [n.get_content() for n in nos_chunk]
    vecs_nos = modelo_embeddings.encode(textos_nos)
    vec_pergunta = modelo_embeddings.encode([PERGUNTA_TESTE])[0]
    melhor = int(np.argmax(vecs_nos @ vec_pergunta))
    resultados_config[nome_config] = {
        "n_chunks": len(nos_chunk),
        "comprimento_medio": np.mean([len(t) for t in textos_nos]),
        "melhor_chunk": textos_nos[melhor],
        "score": float((vecs_nos @ vec_pergunta)[melhor]),
    }

print(f"PERGUNTA: {PERGUNTA_TESTE}\n")
for nome_config, r in resultados_config.items():
    print(f"── {nome_config}: {r['n_chunks']} chunks (média {r['comprimento_medio']:.0f} caracteres)")
    print(f"   Melhor chunk [{r['score']:.3f}]:")
    conteudo = r["melhor_chunk"]
    print("   " + (conteudo.replace(chr(10), chr(10) + "   ")))
    print()

### 5.5 Discussão: não existe chunking universal

A configuração depende do **documento** e das **perguntas**:

| Tipo de documento | Estratégia sugerida |
|---|---|
| **Normativas / regulamentos** | chunks por artigo ou inciso (estrutura); overlap baixo |
| **Entrevistas / história oral** | chunks maiores (uma ideia pode ocupar vários turnos de fala); overlap alto |
| **FAQs** | um chunk por pergunta-resposta (estrutura perfeita para RAG) |
| **Atas / minutas** | por ponto da pauta |

> 🧪 **Experimente isto:** troque `PERGUNTA_TESTE` por uma pergunta cuja resposta precise de *contexto* (p. ex. "quais materiais não saem da sala de leitura?") e veja qual configuração ganha agora.

---
## 6. Índice vetorial in-memory com LlamaIndex

### 6.1 De peças soltas a framework

Até aqui construímos tudo à mão: ler arquivos, fazer chunking, gerar embeddings, buscar com numpy. O **LlamaIndex** empacota exatamente esse fluxo em três abstrações:

- **`Document`**: um arquivo/fonte com sua metadata.
- **`Node`**: um chunk com metadata e relações (o que o nosso `SentenceSplitter` gerava).
- **`VectorStoreIndex`**: o índice — gera os embeddings dos nodes e resolve a busca top-k.

O importante: **já sabemos o que cada peça faz por dentro**, então o framework deixa de ser uma caixa-preta.

### 6.2 Construir o índice

Conectamos o **nosso** modelo de embeddings do Hugging Face (nada de APIs pagas) através de `Settings`, a configuração global do LlamaIndex.

In [ ]:
# ═══ 6.2 Construir o índice in-memory ═══
# Liberamos a cópia "crua" do modelo de embeddings: o LlamaIndex carrega a sua
# (usamos pop para que a célula funcione mesmo se você pulou seções)
for _variavel in ["modelo_embeddings", "vecs_paragrafos", "vecs_corpus", "vecs_armadilha"]:
    globals().pop(_variavel, None)
gc.collect()

from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print(f"⏳ Conectando embeddings do Hugging Face ao LlamaIndex: {NOME_EMBEDDINGS}")
Settings.embed_model = HuggingFaceEmbedding(model_name=NOME_EMBEDDINGS, device=dispositivo)
Settings.llm = None                      # ainda sem LLM: só retrieval (a seção 8 o conecta)
Settings.node_parser = SentenceSplitter(chunk_size=256, chunk_overlap=30)

# file_metadata: adicionamos o idioma como metadata de cada documento
def metadata_arquivo(caminho):
    nome = os.path.basename(caminho)
    return {"file_name": nome, "idioma": "pt" if nome.endswith("_pt.md") else "es"}

documentos = SimpleDirectoryReader(str(PASTA_CORPUS), file_metadata=metadata_arquivo).load_data()
print(f"   {len(documentos)} documentos carregados")

t0 = time.time()
indice = VectorStoreIndex.from_documents(documentos, show_progress=True)
print(f"✅ Índice construído em {time.time() - t0:.1f}s")

### 6.3 Retrieval básico

`as_retriever(similarity_top_k=k)` nos dá o buscador. Vamos inspecionar os **nós recuperados e seus scores** — é o equivalente exato do nosso `buscar()` caseiro.

In [ ]:
# ═══ 6.3 Retrieval e inspeção de nós ═══
recuperador = indice.as_retriever(similarity_top_k=3)

def mostrar_nos(lista_nos):
    for n in lista_nos:
        print(f"[score {n.score:.3f}] {n.metadata.get('file_name')} (idioma: {n.metadata.get('idioma')})")
        conteudo = " ".join(n.get_content().split())
        print(f"   {conteudo}\n")

nos = recuperador.retrieve("Em que dias tem reforço escolar?")
mostrar_nos(nos)

### 6.4 Exercício: 5 perguntas, avaliação a olho

▶️ Vamos fazer 5 perguntas de tipos diferentes e avaliar **a olho** se os chunks recuperados servem para respondê-las. Este hábito — olhar o que o retrieval traz **antes** de conectar o LLM — é a ferramenta de debugging mais importante de RAG.

In [ ]:
# ═══ 6.4 Cinco perguntas ao índice ═══
perguntas_exercicio = [
    "Quantos livros posso levar por vez?",                   # direta, no corpus (doc ES)
    "¿Qué decidió la asamblea sobre el horario?",            # em espanhol, doc ES
    "Posso imprimir peças em 3D na biblioteca?",             # paráfrase, não literal
    "Quando acontece a oficina de compostagem?",             # doc em português
    "Qual é o orçamento anual da biblioteca?",               # NÃO está no corpus
]

for pergunta in perguntas_exercicio:
    print("─" * 70)
    print(f"❓ {pergunta}\n")
    mostrar_nos(recuperador.retrieve(pergunta))

print("─" * 70)
print("""📝 O que observar:
  - a última pergunta NÃO tem resposta no corpus... mas o retriever devolve
    'algo' mesmo assim (os k vizinhos mais próximos SEMPRE existem).
    Detectar esse caso é trabalho do LLM + prompt (seções 8 e 10)""")

### 6.5 Persistência simples

O índice in-memory vive na RAM: se você reiniciar o kernel, é preciso gerar todos os embeddings de novo. O `StorageContext` permite **salvá-lo no disco e recarregá-lo**.

In [ ]:
# ═══ 6.5 Salvar e recarregar o índice ═══
from llama_index.core import StorageContext, load_index_from_storage

PASTA_INDICE = "indice_salvo"

indice.storage_context.persist(persist_dir=PASTA_INDICE)
arquivos_salvos = os.listdir(PASTA_INDICE)
print(f"💾 Índice salvo em '{PASTA_INDICE}/': {arquivos_salvos}")

# Recarregar (assim começaria uma sessão nova, sem gerar os embeddings do corpus de novo)
contexto_armazenamento = StorageContext.from_defaults(persist_dir=PASTA_INDICE)
indice_recarregado = load_index_from_storage(contexto_armazenamento)

nos = indice_recarregado.as_retriever(similarity_top_k=1).retrieve("horário da biblioteca")
print(f"\n✅ Índice recarregado e funcionando: [{nos[0].score:.3f}] {nos[0].metadata.get('file_name')}")

### 6.6 Quando o in-memory é suficiente

- ✅ Corpus pequeno (centenas ou poucos milhares de chunks)
- ✅ Protótipo / experimentação / este workshop
- ✅ Um único processo acessando o índice
- ❌ Corpus grande, múltiplos usuários, filtros complexos, atualizações frequentes → **seção 7 (Qdrant)**

🟢 Se você está no caminho de exploração simples, pode pular direto para a **seção 8**.

---
## 7. Qdrant: banco de dados vetorial

### 7.1 Por que um banco vetorial

O índice in-memory recalcula e recarrega tudo na RAM. Um **banco de dados vetorial** como o Qdrant oferece:

- **Escala**: milhões de vetores com busca aproximada em milissegundos.
- **Persistência real**: os vetores vivem no disco, com transações.
- **Filtros**: "busque só em documentos de 2024", "só em espanhol".
- **Múltiplas coleções**: um corpus por projeto, mesmo banco.

### 7.2 Setup: modo embutido

`QdrantClient(path="...")` cria um banco **local embutido** — um diretório no disco, sem servidor nem Docker. Para produção com múltiplos processos usa-se o servidor, mas a API é a mesma.

### 7.3 Conceitos

| Conceito Qdrant | Equivalente no que já fizemos |
|---|---|
| **Coleção** | nosso índice (com sua métrica de distância: cosseno, claro) |
| **Ponto** | um chunk com embedding (vetor + id) |
| **Payload** | a metadata que viaja com o chunk (`file_name`, `idioma`) |

In [ ]:
# ═══ 7.2-7.4 Qdrant embutido como backend do mesmo VectorStoreIndex ═══
from qdrant_client import QdrantClient
from llama_index.vector_stores.qdrant import QdrantVectorStore

CAMINHO_QDRANT = "qdrant_dados"
NOME_COLECAO = "biblioteca_arroyo_claro"

cliente_qdrant = QdrantClient(path=CAMINHO_QDRANT)  # banco embutido: um diretório no disco

# Se reexecutarmos o notebook, apagamos a coleção para não duplicar pontos
if cliente_qdrant.collection_exists(NOME_COLECAO):
    cliente_qdrant.delete_collection(NOME_COLECAO)

armazem_vetorial = QdrantVectorStore(client=cliente_qdrant, collection_name=NOME_COLECAO)
contexto_qdrant = StorageContext.from_defaults(vector_store=armazem_vetorial)

# Mesmo código da seção 6 — só muda o storage_context. Essa é a graça:
# migrar de in-memory para banco vetorial sem tocar no resto do pipeline.
t0 = time.time()
indice_qdrant = VectorStoreIndex.from_documents(
    documentos, storage_context=contexto_qdrant, show_progress=True
)
print(f"✅ Corpus indexado no Qdrant em {time.time() - t0:.1f}s")

info = cliente_qdrant.get_collection(NOME_COLECAO)
config_vetores = info.config.params.vectors
if isinstance(config_vetores, dict):  # coleções com vetores "nomeados"
    config_vetores = list(config_vetores.values())[0]
print(f"   Coleção '{NOME_COLECAO}': {info.points_count} pontos")
print(f"   Métrica de distância: {config_vetores.distance}")
print(f"   Salva no disco em: {CAMINHO_QDRANT}/")

### 7.5 Filtros por metadata

Aqui aparece o primeiro superpoder que o índice in-memory não tinha: **buscar apenas dentro de um subconjunto** definido por metadata — uma fonte, uma data, uma categoria... ou um idioma.

In [ ]:
# ═══ 7.5 Retrieval com filtro por metadata ═══
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters, FilterOperator

pergunta_filtro = "que atividades acontecem no pátio?"

# Sem filtro
print("SEM filtro:")
mostrar_nos(indice_qdrant.as_retriever(similarity_top_k=2).retrieve(pergunta_filtro))

# Com filtro: SOMENTE documentos em espanhol
filtro_es = MetadataFilters(filters=[
    MetadataFilter(key="idioma", value="es", operator=FilterOperator.EQ)
])
print("COM filtro idioma == 'es':")
recuperador_es = indice_qdrant.as_retriever(similarity_top_k=2, filters=filtro_es)
mostrar_nos(recuperador_es.retrieve(pergunta_filtro))

# 🧪 Experimente isto: filtre por file_name == 'acta_asamblea_2024.md' e pergunte
# pelo horário: garante que a resposta saia da ata e não de outro documento

### 7.6 Exercício: comparar Qdrant vs in-memory

Mesmo corpus, mesmos embeddings, mesma métrica → deveriam recuperar (quase) a mesma coisa. Vamos verificar.

In [ ]:
# ═══ 7.6 In-memory vs Qdrant: recuperam a mesma coisa? ═══
perguntas_comparacao = [
    "quantos computadores tem a sala digital?",
    "como funciona a troca de sementes?",
    "¿cuáles son las reglas de préstamo?",
]

for pergunta in perguntas_comparacao:
    nos_memoria = indice.as_retriever(similarity_top_k=2).retrieve(pergunta)
    nos_qdrant = indice_qdrant.as_retriever(similarity_top_k=2).retrieve(pergunta)
    print(f"❓ {pergunta}")
    for rotulo, lista_nos in [("in-memory", nos_memoria), ("qdrant   ", nos_qdrant)]:
        fontes = [f"{n.metadata.get('file_name')} ({n.score:.3f})" for n in lista_nos]
        print(f"   {rotulo} → {fontes}")
    print()

print("📝 Os scores podem diferir nas casas decimais (implementações diferentes do")
print("   mesmo cálculo), mas as fontes recuperadas deveriam coincidir.")

---
## 8. Pipeline RAG completo

Chegou a hora de conectar a última peça: o **LLM gerador**. Retrieval + prompt + geração = RAG completo.

### 8.1 Conectar o LLM local

Carregamos o modelo escolhido na seção 1.4 (`Qwen3-1.7B` ou `Qwen3-0.6B`) via Hugging Face e o conectamos ao LlamaIndex com a integração `HuggingFaceLLM`. Tudo roda **local**: nenhuma pergunta nem documento sai da sua máquina.

In [ ]:
# ═══ 8.1a Carregar o LLM gerador ═══
from transformers import AutoTokenizer, AutoModelForCausalLM

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()  # silenciar avisos informativos de geração

print(f"⏳ Carregando LLM: {nome_llm}")
print("   (na primeira vez baixa os pesos, pode demorar vários minutos)")

tokenizador = AutoTokenizer.from_pretrained(nome_llm)
modelo_llm = AutoModelForCausalLM.from_pretrained(nome_llm, dtype="auto")
modelo_llm = modelo_llm.to(dispositivo).eval()
n_parametros = sum(p.numel() for p in modelo_llm.parameters())
print(f"✅ LLM carregado: {n_parametros/1e9:.1f}B parâmetros em '{dispositivo}'")

In [ ]:
# ═══ 8.1b Conectar o LLM ao LlamaIndex ═══
from llama_index.llms.huggingface import HuggingFaceLLM

SYSTEM_PROMPT_RAG = (
    "Você responde perguntas sobre os documentos da Biblioteca Comunitária Arroyo Claro. "
    "Responda de forma breve e precisa, usando somente as informações do contexto dado. "
    "Se a informação não estiver no contexto, diga que não a encontrou nos documentos."
)

def prompt_de_chat(texto_prompt):
    """Converte o prompt plano que o LlamaIndex monta para o formato de chat do
    modelo (com o nosso system prompt, e sem modo raciocínio)."""
    mensagens = [
        {"role": "system", "content": SYSTEM_PROMPT_RAG},
        {"role": "user", "content": texto_prompt},
    ]
    return tokenizador.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

Settings.llm = HuggingFaceLLM(
    model=modelo_llm,                  # o modelo que já carregamos em 8.1a
    tokenizer=tokenizador,
    max_new_tokens=256,                # respostas curtas e diretas
    completion_to_prompt=prompt_de_chat,
    generate_kwargs={"do_sample": False,  # determinístico: equivale a temperature=0
                     "pad_token_id": tokenizador.eos_token_id},
)
print("✅ LLM conectado ao LlamaIndex (Settings.llm)")
print("   - do_sample=False → geração determinística (equivale a temperature=0)")
print("   - max_new_tokens=256 → respostas curtas e diretas")

### 8.2 Query engine: RAG em uma linha

`as_query_engine()` junta tudo: **retrieval** (top-k chunks) + **prompt** (template com o contexto injetado) + **geração**.

In [ ]:
# ═══ 8.2 Primeiro pipeline RAG completo ═══
motor_consultas = indice.as_query_engine(similarity_top_k=3)

t0 = time.time()
resposta = motor_consultas.query("Quantos livros posso levar e por quanto tempo?")
print(f"🤖 RESPOSTA ({time.time() - t0:.1f}s):\n{resposta}\n")

print("📚 FONTES USADAS:")
for no_fonte in resposta.source_nodes:
    print(f"   [{no_fonte.score:.3f}] {no_fonte.metadata.get('file_name')}")

### 8.3 Abrir a caixa: o que o LLM recebe de verdade?

Nada de mágica: o query engine usa um **template de prompt** e injeta nele os chunks recuperados. Vamos vê-lo, e ver também o prompt final concreto da última consulta.

In [ ]:
# ═══ 8.3a O template que o query engine usa ═══
templates = motor_consultas.get_prompts()
print("Templates do query engine:", list(templates.keys()), "\n")
print("═══ text_qa_template (o principal) ═══\n")
print(templates["response_synthesizer:text_qa_template"].get_template())

In [ ]:
# ═══ 8.3b O prompt final, reconstruído à mão ═══
pergunta_demo = "Quantos livros posso levar e por quanto tempo?"
nos_demo = indice.as_retriever(similarity_top_k=3).retrieve(pergunta_demo)
contexto_demo = "\n\n".join(n.get_content() for n in nos_demo)

prompt_final = templates["response_synthesizer:text_qa_template"].format(
    context_str=contexto_demo, query_str=pergunta_demo
)
print("Isto é (essencialmente) o que o LLM recebe:\n")
print("┌" + "─" * 68)
for linha in prompt_final.split("\n"):
    print("│ " + linha)
print("└" + "─" * 68)

**`response_mode`: o que fazer quando o contexto não cabe em um prompt**

| Modo | Estratégia | Quando |
|---|---|---|
| `compact` (default) | concatena todos os chunks na menor quantidade de prompts possível | corpus pequenos, o nosso |
| `refine` | responde com o primeiro chunk e *refina* a resposta chunk a chunk | muitos chunks, respostas longas |
| `tree_summarize` | resume por níveis, como um torneio | resumos de corpus inteiros |

Escolhe-se com `index.as_query_engine(response_mode="refine")`.

### 8.4 Parâmetros que importam

- **`similarity_top_k`**: quantos chunks entram no prompt. Mais nem sempre é melhor: contexto irrelevante distrai o modelo.
- **System prompt**: "responda somente com o contexto dado" — já configuramos em 8.1b, e é a primeira defesa contra alucinações.
- **Temperature baixa / determinística** (`do_sample=False`): para respostas factuais não queremos criatividade.

### 8.5 Primeiro exercício: perguntas dentro e fora do corpus

A prova de fogo de um pipeline RAG honesto: o que ele faz quando a resposta **não está** nos documentos?

In [ ]:
# ═══ 8.5 Dentro vs fora do corpus ═══
perguntas_teste = [
    ("DENTRO do corpus", "Quais variedades de sementes tem o banco de sementes?"),
    ("DENTRO do corpus", "¿Cuándo se reciben donaciones de libros en portugués?"),
    ("FORA do corpus",   "Qual é o orçamento anual da biblioteca?"),
    ("FORA do corpus",   "Quem é a pessoa que preside a comissão diretiva?"),
]

for categoria, pergunta in perguntas_teste:
    print("═" * 70)
    print(f"[{categoria}] ❓ {pergunta}")
    resposta = motor_consultas.query(pergunta)
    print(f"🤖 {resposta}")
    fontes = {n.metadata.get("file_name") for n in resposta.source_nodes}
    print(f"📚 fontes: {fontes}")

print("═" * 70)
print("""📝 O que observar:
  - nas perguntas FORA do corpus, o retrieval traz chunks mesmo assim (os
    k vizinhos mais próximos sempre existem) — o modelo deveria dizer que não
    encontrou a informação. Ele fez isso? Ou 'completou' com algo inventado?
  - esse comportamento depende do system prompt E do tamanho do modelo:
    modelos pequenos se abstêm pior""")

### 8.6 Vá além: template de prompt em português

O template default do LlamaIndex está **em inglês**. O modelo o entende, mas misturar idiomas no prompt pode degradar respostas — e queremos controle total sobre a instrução. Vamos substituí-lo por uma versão em português que além disso peça **abstenção explícita**.

In [ ]:
# ═══ 8.6 Template em português com abstenção explícita ═══
from llama_index.core import PromptTemplate

template_pt = PromptTemplate(
    "As informações de contexto estão a seguir.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Usando somente as informações do contexto (e não seu conhecimento prévio), "
    "responda a pergunta no mesmo idioma em que ela foi formulada.\n"
    "Se a resposta não estiver no contexto, responda exatamente: "
    "'Não encontrei esta informação nos documentos.'\n"
    "Pergunta: {query_str}\n"
    "Resposta: "
)

motor_consultas.update_prompts({"response_synthesizer:text_qa_template": template_pt})
print("✅ Template substituído. Repetimos as perguntas problemáticas:\n")

for pergunta in ["Qual é o orçamento anual da biblioteca?",
                 "Quais variedades de sementes tem o banco de sementes?"]:
    print(f"❓ {pergunta}")
    print(f"🤖 {motor_consultas.query(pergunta)}\n")

# 🧪 Experimente isto: traduza o template para o espanhol e teste com perguntas em ES

---
## 9. Reranking: refinar o que foi recuperado

### 9.1 O problema do top-k

O retrieval por embeddings é **rápido porém grosseiro**: comprime cada texto em um único vetor *antes* de conhecer a pergunta. O mais *similar* nem sempre é o mais *relevante* (discutimos isso em 4.5).

### 9.2 Bi-encoder vs cross-encoder

```
  BI-ENCODER (embeddings, seções 3-8)         CROSS-ENCODER (reranker)

  pergunta ──→ [vetor]  ┐                     ┌──────────────────────┐
                        ├─→ cosseno           │  pergunta + chunk    │ ──→ score
  chunk ─────→ [vetor]  ┘                     │  (lidos JUNTOS)      │
                                              └──────────────────────┘
  ✔ rapidíssimo: vetores pré-calculados       ✔ muito mais preciso: vê a
  ✘ cada texto é comprimido "às cegas",         interação pergunta-chunk
    sem conhecer a pergunta                   ✘ caro: um forward pass para
                                                cada par (pergunta, chunk)
```

A estratégia padrão é um **funil**: o bi-encoder filtra milhões → top-20 baratos; o cross-encoder reordena esses 20 → top-5 bons. O caro é aplicado só sobre o pouco.

### 9.3 Implementação

`Qwen3-Reranker-0.6B` recebe o par (pergunta, chunk) e devolve a probabilidade de o chunk ser relevante. Nós o integramos como **node postprocessor** do LlamaIndex: um passo que transforma os nós recuperados antes da geração.

In [ ]:
# ═══ 9.3a Carregar o reranker ═══
print(f"⏳ Carregando reranker: {NOME_RERANKER}")
tokenizador_rr = AutoTokenizer.from_pretrained(NOME_RERANKER, padding_side="left")
modelo_rr = AutoModelForCausalLM.from_pretrained(NOME_RERANKER, dtype="auto").to(dispositivo).eval()

# O reranker da Qwen é um LLM que responde "yes"/"no" à pergunta
# "este documento responde esta consulta?" — o score é P("yes")
ID_SIM = tokenizador_rr.convert_tokens_to_ids("yes")
ID_NAO = tokenizador_rr.convert_tokens_to_ids("no")

PREFIXO_RR = (
    "<|im_start|>system\nJudge whether the Document meets the requirements based on "
    "the Query and the Instruct provided. Note that the answer can only be \"yes\" or "
    "\"no\".<|im_end|>\n<|im_start|>user\n"
)
SUFIXO_RR = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
INSTRUCAO_RR = "Given a web search query, retrieve relevant passages that answer the query"

def pontuar_relevancia(pergunta, texto_chunk):
    """Devolve P(relevante) segundo o cross-encoder, entre 0 e 1."""
    entrada = (f"{PREFIXO_RR}<Instruct>: {INSTRUCAO_RR}\n"
               f"<Query>: {pergunta}\n<Document>: {texto_chunk}{SUFIXO_RR}")
    tokens = tokenizador_rr(entrada, return_tensors="pt", truncation=True,
                            max_length=2048).to(dispositivo)
    with torch.no_grad():
        logits = modelo_rr(**tokens).logits[0, -1]
    return torch.softmax(torch.stack([logits[ID_NAO], logits[ID_SIM]]), dim=0)[1].item()

# Teste rápido
p = "quantos livros posso levar?"
print(f"\n'{p}' vs chunk do regulamento:  {pontuar_relevancia(p, 'Cada persona puede llevarse hasta 3 libros a la vez, por 21 días.'):.3f}")
print(f"'{p}' vs chunk de compostagem:  {pontuar_relevancia(p, 'A oficina de compostagem acontece na última terça-feira do mês.'):.3f}")

In [ ]:
# ═══ 9.3b Integrá-lo como node postprocessor do LlamaIndex ═══
from typing import List, Optional
from llama_index.core.postprocessor.types import BaseNodePostprocessor
from llama_index.core.schema import NodeWithScore, QueryBundle

class RerankerQwen(BaseNodePostprocessor):
    """Reordena os nós com o cross-encoder, filtra por limiar e devolve top_n."""
    top_n: int = 3
    limiar: float = 0.5   # P(relevante) mínima para entrar no contexto

    def _postprocess_nodes(
        self, nodes: List[NodeWithScore], query_bundle: Optional[QueryBundle] = None
    ) -> List[NodeWithScore]:
        for no_atual in nodes:
            no_atual.score = pontuar_relevancia(query_bundle.query_str, no_atual.get_content())
        ordenados = sorted(nodes, key=lambda n: n.score, reverse=True)
        filtrados = [n for n in ordenados if n.score >= self.limiar]
        return (filtrados or ordenados[:1])[: self.top_n]  # nunca deixar o contexto vazio

# O funil: recuperar 10 baratos com embeddings, e ficar somente com os que o
# cross-encoder considera realmente relevantes (até 3)
motor_com_reranker = indice.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[RerankerQwen(top_n=3, limiar=0.5)],
)
motor_com_reranker.update_prompts({"response_synthesizer:text_qa_template": template_pt})
print("✅ Query engine com reranking: top-10 por embeddings → cross-encoder → limiar 0.5")
print("   Detalhe-chave: os scores do cross-encoder são probabilidades, então")
print("   PODEMOS filtrar por limiar. Com a similaridade cosseno não há limiar universal:")
print("   0.3 pode ser 'muito relevante' em um corpus e ruído em outro.")

### 9.4 Exercício: com e sem reranking, mesma pergunta

Vamos usar uma pergunta com **distratores**: *"Quais materiais não são emprestados a domicílio?"*. A resposta está no regulamento (os materiais de referência não saem da sala de leitura), mas o documento de compostagem também lista "o que não pode ir" — um vizinho temático que **soa** parecido sem responder a pergunta.

Vamos comparar quais chunks chegam ao LLM, **com quais scores**, e quanto custa em latência.

In [ ]:
# ═══ 9.4 Comparação com/sem reranking ═══
pergunta_dificil = "Quais materiais não são emprestados a domicílio?"

# Sem reranking: top-3 direto por embeddings
t0 = time.time()
resposta_sem = motor_consultas.query(pergunta_dificil)
lat_sem = time.time() - t0

# Com reranking: top-10 → cross-encoder → top-3
t0 = time.time()
resposta_com = motor_com_reranker.query(pergunta_dificil)
lat_com = time.time() - t0

print(f"❓ {pergunta_dificil}\n")
print(f"── SEM reranking ({lat_sem:.1f}s) — scores = similaridade cosseno:")
for n in resposta_sem.source_nodes:
    print(f"   [{n.score:.3f}] {n.metadata.get('file_name')}")
print(f"🤖 {resposta_sem}\n")
print(f"── COM reranking ({lat_com:.1f}s) — scores = P(relevante) do cross-encoder:")
for n in resposta_com.source_nodes:
    print(f"   [{n.score:.3f}] {n.metadata.get('file_name')}")
print(f"🤖 {resposta_com}\n")

print("📝 O que aconteceu aqui:")
print("   - por cosseno os chunks parecem igualmente 'próximos' (~0.2-0.3), então")
print("     o contexto se enche de vizinhos temáticos que NÃO respondem — e esse")
print("     ruído pode confundir o modelo (o 'mais nem sempre é melhor' de 8.4)")
print("   - o cross-encoder, que lê pergunta e chunk JUNTOS, é categórico:")
print("     regulamento com score altíssimo, distratores perto de zero → o limiar")
print("     os deixa de fora e o LLM recebe um contexto pequeno porém limpo")

In [ ]:
# ═══ 9.4b Visualizar o custo em latência ═══
fig = go.Figure(go.Bar(
    x=["sem reranking", "com reranking"],
    y=[lat_sem, lat_com],
    marker=dict(color=CORES[0], cornerradius=4), width=0.5,
    text=[f"{lat_sem:.1f}s", f"{lat_com:.1f}s"], textposition="outside",
    hovertemplate="%{x}: %{y:.2f}s<extra></extra>",
))
estilizar_figura(fig, "Latência da consulta completa (retrieval + geração)")
fig.update_layout(yaxis_title="segundos", height=380)
fig.show()

print("📝 O custo extra do reranking = 10 forward passes do cross-encoder.")
print("   Em corpus pequenos pode não mudar muito a resposta; seu valor aparece")
print("   com corpus grandes, perguntas ambíguas e quando o erro custa caro.")

### 9.5 Discussão: quando o custo se justifica?

- **Corpus grande**: com 100 chunks, o top-3 por embeddings costuma bastar; com 100.000, o funil top-50 → top-5 muda tudo.
- **Perguntas ambíguas**: negações, condições, perguntas que misturam temas.
- **Alto custo do erro**: normativas, saúde, decisões — vale pagar segundos extras por precisão.
- **Orçamento de contexto**: com um LLM pequeno, mandar 3 chunks *bons* rende mais do que mandar 10 medíocres.

---
## 10. Citações e redução de alucinações

### 10.1 Alucinações em RAG

RAG **reduz** as alucinações (o modelo tem o contexto correto à mão) mas **não as elimina**: o modelo pode ignorar o contexto, distorcê-lo ou preencher lacunas. Tipos frequentes:

- **Resposta sem sustentação**: afirma algo que não está em nenhuma fonte recuperada.
- **Mistura de fontes**: combina dados de dois documentos em uma afirmação falsa (p. ex. o horário da oficina de horta com o dia do clube de leitura).
- **Excesso de confiança**: o contexto não é suficiente, mas ele responde mesmo assim, com total segurança.

### 10.2 Grounding: amarrar a resposta às fontes

**Grounding** = exigir que cada afirmação saia do contexto recuperado. As duas ferramentas básicas já usamos: system prompt restritivo e **abstenção explícita** ("Não encontrei esta informação nos documentos", seção 8.6). A terceira é estrutural: **citações**.

### 10.3 Citações como mecanismo

O `CitationQueryEngine` do LlamaIndex numera cada chunk como fonte `[1]`, `[2]`, ... e pede ao modelo que cite cada afirmação:

- **A citação é a unidade verificável**: quem lê pode ir à fonte `[2]` e conferir.
- **Citar muda o comportamento do modelo**: tendo que atribuir cada frase, fica mais difícil inventar. A estrutura do prompt empurra para o grounding.

In [ ]:
# ═══ 10.3 CitationQueryEngine com template em português ═══
from llama_index.core.query_engine import CitationQueryEngine

template_citacoes_pt = PromptTemplate(
    "A seguir há fontes numeradas. Responda a pergunta usando somente essas fontes.\n"
    "Escreva a resposta com as informações encontradas (não repita a pergunta) e "
    "adicione o número da fonte depois de cada afirmação, por exemplo [1].\n"
    "Exemplo de resposta bem formada:\n"
    "Pergunta: De que cor fica o céu ao entardecer?\n"
    "Resposta: Ao entardecer o céu pode ficar avermelhado [2].\n"
    "Se a resposta não estiver nas fontes, responda: "
    "'Não encontrei esta informação nos documentos.'\n"
    "Fontes:\n"
    "------\n"
    "{context_str}\n"
    "------\n"
    "Pergunta: {query_str}\n"
    "Resposta com citações: "
)

motor_citacoes = CitationQueryEngine.from_args(
    indice,
    similarity_top_k=3,
    citation_chunk_size=256,            # re-divide os chunks em fontes citáveis mais finas
    citation_qa_template=template_citacoes_pt,
)
print("✅ Query engine com citações pronto")

### 10.4 Exercício: mesma pergunta, com e sem citações

E o passo-chave, que nenhuma ferramenta faz por nós: **verificar manualmente** se cada citação realmente sustenta a afirmação.

In [ ]:
# ═══ 10.4 Query engine normal vs com citações ═══
pergunta_citacoes = "O que a assembleia de 2024 decidiu sobre o horário e a comissão?"

print(f"❓ {pergunta_citacoes}\n")
print("── SEM citações:")
print(f"🤖 {motor_consultas.query(pergunta_citacoes)}\n")

print("── COM citações:")
resposta_citada = motor_citacoes.query(pergunta_citacoes)
print(f"🤖 {resposta_citada}\n")

print("📚 FONTES NUMERADAS (para verificar à mão):")
for i, no_fonte in enumerate(resposta_citada.source_nodes, start=1):
    conteudo = " ".join(no_fonte.get_content().split())
    print(f"\n[{i}] ({no_fonte.metadata.get('file_name')})")
    print(f"    {conteudo[:220]}{'...' if len(conteudo) > 220 else ''}")

In [ ]:
# ═══ 10.4b Verificação manual guiada ═══
print("""🔎 EXERCÍCIO DE VERIFICAÇÃO (à mão, em grupo):

  1. Tome cada afirmação da resposta citada acima.
  2. Vá à fonte [n] que a acompanha e verifique:
       A fonte realmente diz isso?              → citação CORRETA
       A fonte fala de outra coisa?             → citação MAL ATRIBUÍDA
       A afirmação não aparece em NENHUMA?      → citação INVENTADA
  3. Preste atenção especial aos números (horários, quantidades, anos):
     são o lugar clássico onde o modelo mistura fontes.

  🧪 Experimente isto: faça uma pergunta cuja resposta combine DOIS documentos
  ("que atividades acontecem às terças e o que foi decidido sobre o horário?")
  e verifique se cada metade da resposta cita o documento correto.""")

### 10.5 Estratégias complementares

As citações não substituem o resto do arsenal — elas se somam:

- **System prompt restritivo** ("responda somente com o contexto") — seção 8.4.
- **Geração determinística** (`do_sample=False` / `temperature=0`) para respostas factuais — seção 8.1b.
- **Mostrar sempre os chunks fonte junto com a resposta** na interface: mesmo que o modelo não cite, quem usa o sistema pode verificar. É a mais simples e a mais robusta.

### 10.6 Discussão: rastreabilidade como prática de cuidado

- **Quem verifica** as respostas antes de chegarem a outras pessoas? O sistema mostra as fontes para que verificar seja possível?
- **Quem responde pelo erro** se o sistema der uma informação equivocada (um horário, um requisito, um direito)?
- **Qual erro seria mais grave** no seu projeto: inventar informação, ou omitir informação que estava lá?

Para sistemas comunitários, a resposta citada não é um luxo técnico: é o que permite que a confiança não dependa de "acreditar na máquina".

---
## 11. Avaliação de RAG

### 11.1 Duas coisas diferentes a avaliar

Um pipeline RAG pode falhar em dois lugares diferentes, e convém medi-los separadamente:

1. **Retrieval**: recuperamos os chunks corretos? (se aqui falha, todo o resto falha)
2. **Geração**: dado um bom contexto, a resposta é **fiel** ao contexto e **responde** a pergunta?

### 11.2 Avaliar retrieval: hit rate

Montamos um **mini conjunto de avaliação**: perguntas + o documento onde está a resposta. A métrica mais simples é o **hit rate@k**: o documento correto apareceu entre os k chunks recuperados?

Um detalhe importante: incluímos perguntas **diretas** (usam palavras do documento) e perguntas **difíceis** (paráfrase, formulação indireta, ou perguntar em português algo que está em um documento em espanhol). Se o conjunto só tem perguntas fáceis, tudo dá 100% e a avaliação não discrimina entre configurações.

In [ ]:
# ═══ 11.2a Mini conjunto de avaliação: pergunta → documento esperado ═══
# Metade diretas, metade difíceis (paráfrase, pergunta indireta, cruzamento de
# idiomas): se todas as perguntas são fáceis, a métrica dá 100% e não
# aprendemos nada sobre o pipeline.
conjunto_avaliacao = [
    # diretas
    ("O que não pode ir na composteira?",                       "oficina_compostagem_pt.md"),
    ("Posso doar mais de cinquenta livros de uma vez?",         "acervo_emprestimos_pt.md"),
    ("Que dia é a oficina de xadrez?",                          "talleres_y_actividades.md"),
    ("Como reservo a impressora 3D?",                           "equipamiento_sala_digital.md"),
    # difíceis
    ("O que acontece se eu me atrasar na devolução?",           "reglamento_prestamos.md"),
    ("Onde a biblioteca funcionava no início?",                 "historia_biblioteca.md"),
    ("Quem cuida da biblioteca no dia a dia?",                  "historia_biblioteca.md"),
    ("Para onde vai o dinheiro arrecadado na feira do livro?",  "acta_asamblea_2024.md"),
]

def hit_rate(indice_aval, k):
    """Proporção de perguntas cujo documento esperado aparece no top-k."""
    acertos = 0
    recuperador_aval = indice_aval.as_retriever(similarity_top_k=k)
    for pergunta, doc_esperado in conjunto_avaliacao:
        fontes = [n.metadata.get("file_name") for n in recuperador_aval.retrieve(pergunta)]
        acertos += doc_esperado in fontes
    return acertos / len(conjunto_avaliacao)

for k in [1, 3, 5]:
    taxa = hit_rate(indice, k)
    print(f"hit rate@{k}: {taxa:.0%}  {'█' * int(taxa * 30)}")

In [ ]:
# ═══ 11.2b O efeito de chunk_size e top_k sobre o retrieval ═══
print("⏳ Reindexando o corpus com dois chunk_size diferentes (corpus pequeno, é rápido)...")

indices_por_chunk = {}
for tamanho in [96, 512]:
    splitter = SentenceSplitter(chunk_size=tamanho, chunk_overlap=20)
    nos_aval = splitter.get_nodes_from_documents(documentos)
    indices_por_chunk[tamanho] = VectorStoreIndex(nos_aval)

valores_k = [1, 2, 3, 5]
fig = go.Figure()
for (tamanho, indice_aval), cor in zip(indices_por_chunk.items(), CORES):
    taxas = [hit_rate(indice_aval, k) for k in valores_k]
    fig.add_trace(go.Scatter(
        x=valores_k, y=taxas, mode="lines+markers",
        name=f"chunk_size={tamanho}",
        line=dict(color=cor, width=2), marker=dict(size=9),
        hovertemplate="top_k=%{x}: %{y:.0%}<extra>chunk_size=" + str(tamanho) + "</extra>",
    ))

estilizar_figura(fig, "Hit rate do retrieval segundo top_k e chunk_size")
fig.update_layout(xaxis_title="similarity_top_k", yaxis_title="hit rate",
                  yaxis_tickformat=".0%", yaxis_range=[0, 1.05], height=420)
fig.update_xaxes(tickvals=valores_k)
fig.show()

print("📝 Aumentar top_k sempre ajuda o hit rate... mas coloca mais contexto (e mais")
print("   ruído) no prompt. O chunk_size ótimo depende do corpus: por isso ele")
print("   se mede em vez de se adivinhar.")

### 11.3 Avaliar geração: faithfulness e relevância

- **Faithfulness (fidelidade)**: cada afirmação da resposta se sustenta no contexto recuperado?
- **Relevância**: a resposta responde *o que foi perguntado*?

A forma industrial de medi-las é **LLM-as-judge** (outro LLM maior avalia cada resposta — o LlamaIndex traz `FaithfulnessEvaluator` e `RelevancyEvaluator`). Com modelos pequenos e locais como os nossos, o juiz não é confiável; uma verificação simples e honesta na nossa escala: conferir se os **dados concretos** da resposta (números, dias, anos) aparecem no contexto usado.

In [ ]:
# ═══ 11.3 Verificação simples de fidelidade: os dados concretos da resposta
#     aparecem no contexto que o modelo usou? ═══
def dados_concretos(texto):
    """Extrai números e cifras de um texto (os candidatos clássicos a alucinação)."""
    return set(re.findall(r"\d+(?:[:.]\d+)?", texto))

perguntas_geracao = [
    "Até quantas horas por semana posso reservar a impressora 3D?",
    "Quantas pessoas integram a comissão diretiva depois da assembleia de 2024?",
]

for pergunta in perguntas_geracao:
    resposta = motor_consultas.query(pergunta)
    contexto_usado = " ".join(n.get_content() for n in resposta.source_nodes)
    sem_sustento = dados_concretos(str(resposta)) - dados_concretos(contexto_usado)
    print(f"❓ {pergunta}")
    print(f"🤖 {resposta}")
    if sem_sustento:
        print(f"⚠️ Dados na resposta que NÃO aparecem no contexto: {sem_sustento}")
        print("   (candidatos a alucinação → verificar à mão)")
    else:
        print("✅ Todos os números da resposta estão no contexto recuperado")
    print()

print("📝 Esta checagem é deliberadamente simples: pega números inventados, não")
print("   distorções sutis. Para isso: verificação humana (10.4) ou LLM-as-judge")
print("   com um modelo maior (🚀 llama_index.core.evaluation)")

### 11.4 Avaliação qualitativa por equipe

As métricas automáticas não capturam o que mais importa em um sistema comunitário. Rubrica sugerida para avaliar em equipe (escala 1–4 por dimensão):

| Dimensão | Pergunta-guia |
|---|---|
| **Utilidade** | A resposta serve para a pessoa que perguntou? |
| **Clareza** | Dá para entender sem conhecimento técnico? |
| **Tom** | É apropriado para a comunidade que vai usar o sistema? |
| **Vieses** | Responde igualmente bem para todos os temas/idiomas do corpus? |
| **Honestidade** | Diz "não sei" quando é o caso, ou inventa? |

### 11.5 Exercício integrador: comparar 2 configurações

Vamos fechar medindo de ponta a ponta: **mesma bateria de perguntas, dois pipelines diferentes** (chunks pequenos + top_k alto vs chunks grandes + top_k baixo).

In [ ]:
# ═══ 11.5 Duas configurações, mesmo conjunto de perguntas ═══
config_A = {"nome": "chunks 96 / top_k 5", "indice": indices_por_chunk[96], "k": 5}
config_B = {"nome": "chunks 512 / top_k 2", "indice": indices_por_chunk[512], "k": 2}

print(f"{'':<42} {'hit rate':>9}")
taxas_finais = {}
for config in [config_A, config_B]:
    taxa = hit_rate(config["indice"], config["k"])
    taxas_finais[config["nome"]] = taxa
    print(f"{config['nome']:<42} {taxa:>8.0%}")

# E uma comparação qualitativa das respostas geradas
pergunta_integradora = "Quando a biblioteca foi fundada e quantos livros ela tem hoje?"
print(f"\n❓ {pergunta_integradora}\n")
for config in [config_A, config_B]:
    motor_config = config["indice"].as_query_engine(similarity_top_k=config["k"])
    motor_config.update_prompts({"response_synthesizer:text_qa_template": template_pt})
    t0 = time.time()
    resposta = motor_config.query(pergunta_integradora)
    print(f"── {config['nome']} ({time.time() - t0:.1f}s):")
    print(f"🤖 {resposta}\n")

# 🧪 Experimente isto: adicione uma config com reranking (seção 9) à comparação.
# O hit rate melhora o suficiente para justificar a latência extra?

### 11.6 Discussão: a avaliação nunca está terminada

O mini conjunto de 8 perguntas nos serviu para o workshop, mas as perguntas reais da comunidade vão ser outras. A prática saudável:

- **Registrar as perguntas reais** (com consentimento) e somá-las ao conjunto de avaliação.
- **Revisar periodicamente** as respostas com pior feedback.
- Tratar a avaliação como **monitoramento contínuo**, não como uma prova que se passa uma única vez.

---
## 12. Aplicação a projetos

### 12.1 Definir o caso de uso

Antes de tocar em código, três perguntas com sua equipe:

1. **Quais documentos?** Estão digitalizados? Em que formato e idioma(s)? Há informação sensível que não deveria entrar no corpus?
2. **Quais perguntas?** Escrevam 10 perguntas reais que as pessoas fazem. Esse é o primeiro conjunto de avaliação de vocês (seção 11.2).
3. **Quais usuários?** Quem consulta? Que custo tem uma resposta incorreta para essa pessoa?

### 12.2 Escolher configuração

| Decisão | Escolha isto... | ...se |
|---|---|---|
| **Índice** | in-memory (seção 6) | protótipo, corpus pequeno, um único processo |
| | Qdrant (seção 7) | corpus crescente, filtros por metadata, persistência |
| **Reranking** | sem (seção 8) | corpus pequeno, latência importa |
| | com (seção 9) | corpus grande, perguntas ambíguas, erro caro |
| **LLM** | Qwen3-1.7B | GPU ou boa RAM: melhores respostas e abstenção |
| | Qwen3-0.6B | CPU / hardware limitado |

### 12.3 Implementação rápida

Todo o workshop, condensado em uma função. Aponte-a para uma pasta com **seus** documentos e você tem seu pipeline funcionando.

In [ ]:
# ═══ 12.3 Pipeline mínimo reutilizável ═══
def construir_pipeline_rag(pasta_documentos, chunk_size=256, top_k=3, com_reranker=False):
    """Constrói um query engine RAG completo sobre uma pasta de documentos."""
    docs = SimpleDirectoryReader(str(pasta_documentos)).load_data()
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=int(chunk_size * 0.1))
    indice_projeto = VectorStoreIndex(splitter.get_nodes_from_documents(docs))

    pos_processadores = [RerankerQwen(top_n=top_k)] if com_reranker else []
    k_inicial = top_k * 3 if com_reranker else top_k
    motor = indice_projeto.as_query_engine(
        similarity_top_k=k_inicial, node_postprocessors=pos_processadores
    )
    motor.update_prompts({"response_synthesizer:text_qa_template": template_pt})
    print(f"✅ Pipeline pronto: {len(docs)} documentos, chunk_size={chunk_size}, "
          f"top_k={top_k}, reranker={'sim' if com_reranker else 'não'}")
    return motor

# Com o corpus do workshop (troque a pasta por 'meus_documentos' para seu projeto):
PASTA_PROJETO = "meus_documentos" if os.path.isdir("meus_documentos") else PASTA_CORPUS
motor_projeto = construir_pipeline_rag(PASTA_PROJETO)

print(f"\n🤖 {motor_projeto.query('Que atividades regulares a biblioteca oferece e em que dias?')}")

### 12.4 Próximo experimento para cada equipe

Modelo para definir o próximo passo (um por equipe):

> - **Corpus inicial**: [quais 5–10 documentos]
> - **10 perguntas de avaliação**: [as que as pessoas fazem de verdade]
> - **Métrica para olhar primeiro**: hit rate@3 do retrieval (seção 11.2)
> - **Primeira melhoria a testar**: [chunking / prompt no idioma do corpus / reranking / modelo maior]
> - **Qual erro seria mais grave** e como o mitigamos: [abstenção / citações / revisão humana]

Regra de ouro: **melhorar o retrieval antes do gerador** — é mais barato e costuma ser onde está o problema.

---
## 13. Encerramento

### 13.1 Síntese

O que construímos, peça por peça:

1. **Embeddings** (seção 3): texto → vetor; perto no espaço = parecido em significado. Vimos também que o espaço não é igualmente bom para todos os idiomas.
2. **Similaridade** (seção 4): cosseno em 3 linhas de numpy, e retrieval completo em 10 — antes de delegá-lo a um framework.
3. **Chunking** (seção 5): não existe configuração universal; depende do documento e das perguntas.
4. **Índice** (seções 6-7): LlamaIndex in-memory para prototipar; Qdrant embutido para persistência e filtros.
5. **Pipeline RAG** (seção 8): retrieval + prompt com contexto + geração local. Abrimos a caixa e vimos o prompt real.
6. **Reranking** (seção 9): o funil bi-encoder → cross-encoder, quando a precisão o justifica.
7. **Citações** (seção 10): grounding verificável; a rastreabilidade como prática de cuidado.
8. **Avaliação** (seção 11): hit rate para o retrieval, fidelidade para a geração, rubrica qualitativa para o que as métricas não veem.

### 13.2 Próximos passos segundo o caminho

- 🟢 **Exploração simples** → monte o corpus do seu projeto (`meus_documentos/`) e use `construir_pipeline_rag()` (seção 12.3). Depois adicione citações (seção 10).
- 🟡 **Experimentação** → experimente com chunking (seção 5) sobre seus documentos reais e meça o hit rate com suas próprias 10 perguntas (seção 11.2).
- 🔴 **Profundidade** → migre seu índice para o Qdrant com metadata útil (seção 7.5), adicione reranking (seção 9) e automatize a avaliação (🚀 `llama_index.core.evaluation`).
- ⚙️ **Hardware limitado** → todo o pipeline funciona com `Qwen3-0.6B`; se a geração estiver lenta, reduza `similarity_top_k` e `max_new_tokens`, e lembre que o retrieval sozinho (seções 3-7) é leve e já é útil por si só.

**O pipeline é de vocês: roda completo nas suas máquinas, com modelos abertos, sobre seus documentos.**